Run Forcasts for dFADs with dynamical CMEMS models istead of static velocity fields

Can run each month at a time since field for all dFADs is the same. 

1) Collect all dFADs initial possitions for forcasting.
        - Same method as 'run_model.py' 

2) make forecasts for entire period

3) 

In [1]:
import parcels
import geopandas as gpd 
from pathlib import Path
import xarray as xr
from datetime import timedelta
import zarr
import functions.funcs as fad
from importlib import reload
reload(fad)
import pandas as pd
import numpy as np
import shapely as sp

fname = r"..\..\Data\cmems.nc" ### Change the field to cmems
field = xr.open_dataset(fname )

ds = gpd.read_parquet(r"..\..\Data\Mapped_SAT_MI_Cleanedspeeds.parquet")

In [2]:
monthrange = pd.date_range("2024-01-1","2025-01-1", freq= "MS")
month = 2
daterange = pd.date_range(monthrange[month], monthrange[month+1])
startdate = monthrange[month]
enddate = monthrange[month+1] + pd.Timedelta(days = 7)

Monthdaterange = pd.date_range(startdate, enddate - pd.Timedelta(days =7), freq= "D")

dFADs = pd.DataFrame(columns = ["BuoyName","lat", "lon", "TimeStamp", "x_speed", "y_speed"] )

for day in Monthdaterange:
    endofday = day + pd.Timedelta(days= 1)
    ds_active = fad.querry_date(ds, day) ## All of the active dFADs at this time  #863 total points 
    ds_active = ds_active.reset_index()
    columns = ["TimeStamp", "x_speed", "y_speed"]
    ds_locations = pd.DataFrame(columns = ["BuoyName","lat", "lon", "TimeStamp", "x_speed", "y_speed"])
    for label in columns: 
        longlist, ids = fad.Column_to_List(ds_active, label, idlist = True)
        
        ds_locations[label] = longlist
    lat, lon  = fad.list_of_latlon(ds_active, droplast= False)
    ds_locations["lat"] = lat
    ds_locations["lon"] =lon
    ds_locations["BuoyName"] = ids
    ds_locations.TimeStamp = pd.to_datetime(ds_locations.TimeStamp)
        
    ds_locations = ds_locations.query(f"TimeStamp > @day")
    ds_locations = ds_locations.query(f"TimeStamp < @endofday")

    ## remove duplicates
    ds_locations = ds_locations.drop_duplicates(subset=["lat"], keep="first")
    ds_locations = ds_locations.drop_duplicates(subset=["lon"], keep="first").reset_index(drop = True)
    ds_locations = ds_locations.sort_values('TimeStamp').groupby("BuoyName").first().reset_index()

    ## These are initial locations to make forecasts from
    dFADs = pd.concat([dFADs,ds_locations.reset_index()])
ds_active = fad.querry_date_range(ds, startdate= startdate, enddate= enddate)
dFADs = dFADs.reset_index(drop = True)
dFADs.TimeStamp = pd.to_datetime(dFADs.TimeStamp)
print(f"amount of dFADs: {len(dFADs)}")


amount of dFADs: 513


In [3]:
print(f"making Forecasts from {startdate} -- {enddate}")

## Make the model... 
filenames = {"uo": fname, "vo": fname}
variables  = {"U": "uo", "V": "vo"}
dimensions = {"lat": "latitude", "lon": "longitude", "time" : "time"}
## fix this and make it a non static field
field_t = field.sel(time = slice(startdate, enddate), depth = 15.81007)## IF CMEMS add depth = 15 argument
runtime = enddate - startdate


making Forecasts from 2024-03-01 00:00:00 -- 2024-04-08 00:00:00


In [4]:

# fieldsetperm = parcels.FieldSet.from_netcdf(filenames, variables, dimensions)
fieldset  = parcels.FieldSet.from_xarray_dataset(field_t, variables, dimensions, allow_time_extrapolation= True) 
fieldset.add_constant("halo_west", fieldset.U.grid.lon[0])
fieldset.add_constant("halo_east", fieldset.U.grid.lon[-1])
fieldset.add_constant("halo_north", fieldset.U.grid.lat[-1])
fieldset.add_constant("halo_south", fieldset.U.grid.lat[0])
fieldset.add_periodic_halo(zonal = True , meridional= True)

def boundryCondition(particle, fieldset,time):
    if particle.lon < fieldset.halo_west or particle.lon > fieldset.halo_east:
        particle.delete()
    if particle.lat < fieldset.halo_south or particle.lat > fieldset.halo_north:
        particle.delete()
        
def Age(particle, fieldset, time):
    particle.age += particle.dt / 3600

### need to add to kill particles after 7 days. 

def end_forcast(particle, fieldset, time): ## need to check if this is working correctly
    if particle.age > 7*24:
        particle.delete()
    
dFADs["timedelta"] = (dFADs.TimeStamp - startdate).dt.total_seconds()

Particles = parcels.JITParticle.add_variable("age", initial = 0) 
Particles = Particles.add_variable("Buoyindex", to_write = 'once')

pset = parcels.ParticleSet.from_list(fieldset, pclass = Particles , lon = dFADs.lon.to_list(), 
                                    lat = dFADs.lat.to_list() , time = dFADs.timedelta.to_list(), Buoyindex = dFADs.index.values) 

output_memorystore = zarr.storage.MemoryStore()
output_file = pset.ParticleFile(name = "TestParticleFile.zarr", outputdt =timedelta(hours= 1))



In [5]:
fieldset.U

<Field>
    name            : 'U'
    grid            : RectilinearZGrid(lon=array([-164.25, -164.17, -164.08, ..., -160.17, -160.08, -160.00], shape=(52,), dtype=float32), lat=array([ 4.00,  4.08,  4.17, ...,  8.08,  8.17,  8.25], shape=(52,), dtype=float32), time=array([ 0.00,  86400.00,  172800.00, ...,  3110400.00,  3196800.00,  3283200.00], shape=(39,)), time_origin=2024-03-01T00:00:00.000000000, mesh='spherical')
    extrapolate time: True
    time_periodic   : False
    gridindexingtype: 'nemo'
    to_write        : False

In [6]:
pset.execute([parcels.AdvectionRK4, boundryCondition, Age, end_forcast], 
            runtime = runtime, ##this should be 8 days 
            dt = timedelta(minutes =5), 
            output_file = output_file, 
            )


INFO: Output files are stored in TestParticleFile.zarr.


c:\Users\czerfass\AppData\Local\miniforge3\envs\parcels\Lib\site-packages\parcels\particleset.py:1127: ParticleSetWarning: Some of the particles have a start time difference that is not a multiple of outputdt. This could cause the first output of some of the particles that start later in the simulation to be at a different time than expected.
  _warn_outputdt_release_desync(outputdt, starttime, self.particledata.data["time_nextloop"])


  0%|          | 300.0/3283200.0 [00:00<19:49, 2760.10it/s]

c:\Users\czerfass\AppData\Local\miniforge3\envs\parcels\Lib\site-packages\parcels\particledata.py:358: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(time - np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\czerfass\AppData\Local\miniforge3\envs\parcels\Lib\site-packages\parcels\particledata.py:359: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.greater_equal(time + np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
c:\Users\czerfass\AppData\Local\miniforge3\envs\parcels\Lib\site-packages\parcels\particledata.py:360: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  | ((np.isnan(pd["dt"])) & np.equal(time, pd["time"], where=np.isfinite(pd["time"])))


100%|██████████| 3283200.0/3283200.0 [11:13<00:00, 4871.62it/s] 


In [42]:
output = xr.open_zarr(r"TestParticleFile.zarr", decode_timedelta=True)
#output = xr.open_zarr(r"..\output\TestParticleFile0.zarr", decode_timedelta=True)
np.set_printoptions(suppress=True, precision=6)
s =  np.datetime64(pd.Timestamp("2024-03-01"))
(output.time.values -s)/np.timedelta64(1, "ns")/1e9

array([[    180.,    3780.,    7380., ...,  597780.,  601380.,  604980.],
       [   3780.,    7380.,   10980., ...,  601380.,  604980.,  605880.],
       [   3780.,    7380.,   10980., ...,  601380.,  604980.,  608280.],
       ...,
       [2692980., 2696580., 2700180., ...,      nan,      nan,      nan],
       [2692980., 2696580., 2700180., ...,      nan,      nan,      nan],
       [2725380., 2728980., 2732580., ...,      nan,      nan,      nan]],
      shape=(513, 169))

In [ ]:

buoy_list = dFADs.BuoyName.tolist()  #### Name are now Nonunique
ds_filtered = ds_active[ds_active["BuoyName"].isin(buoy_list)].reset_index(drop = True) #same as orriginal dFADs but ones weve actually made forcasts for


def Forcast_snippit(ds: gpd.GeoDataFrame, dates, startdate, length)-> gpd.GeoDataFrame: 
    """Grad only the snipbit of dFAD trajectory that lines up with forcast window"""
    ds_s = ds.copy()
    forecast_end = startdate + length
    for i in range(len(ds)): ## Try and grab at the exact start times from dates they should be the same
        timelist = (ds_s.at[i,"TimeStamp"])
        timelist = pd.to_datetime(timelist)
        mask = (timelist >=startdate) & (timelist <= forecast_end)
        timelist = timelist[mask]
        coords = np.asarray(ds.at[i,"geometry"].coords)
        filtered_coords = coords[mask]
        ds_s.at[i,"TimeStamp"] = timelist
        if len(filtered_coords) > 1:
            ds_s.at[i,"geometry"] = sp.geometry.LineString(filtered_coords)
        else: 
            ds_s.at[i,"geometry"] = None
    return ds_s

ds_short_t = Forcast_snippit(ds_filtered, dFADs.TimeStamp, startdate, (enddate -startdate + pd.Timedelta(days = 7))) ## probally start date, definitaly need to change length


In [ ]:
np.set_printoptions(suppress=True, precision=6)
##Saving to CSV file
## Failing on a case where the first point of the dFAD is released at the exact time an output is recorded
dssave = pd.DataFrame(columns = ["BuoyID","Time", "lat_true", "lon_true", "lat_forcast", "lon_forcast", "leadtime"])
output = xr.open_zarr("TestParticleFile.zarr", decode_timedelta=True)
dsout = pd.DataFrame(columns = ["BuoyID","Time", "lat_true", "lon_true", "lat_forcast", "lon_forcast", "leadtime"])
masklarge =~ np.isnan(output.lat[:,:].values)
dFADs_s = dFADs.iloc[output.Buoyindex.values].reset_index(drop = False)
# print(output.Buoyindex.values.size)
# print(dFADs_s.BuoyName)
emptydata = 0
for i, index in enumerate(output.Buoyindex.values): 
    """Take one forcast and matches it with one True Trjactory"""
    print(i)
    id = dFADs_s.BuoyName[i]
    row =  ds_short_t.query("BuoyName == @id").reset_index(drop = True)
    row = row.iloc[0]
    Times= row["TimeStamp"]
    dFAD_times = (Times - startdate).total_seconds() ## convet to seconds since model started 
    mask = masklarge[i,:] 
    forcast_time_start = (output.time[i,:].values[mask])  ## converts to seconds since model has started. 
    forcast_time_start = forcast_time_start/np.timedelta64(1, "s")
    #print(forcast_time_start/3600)
    #print(dFAD_times/3600)
    dFAD_times_s = dFAD_times[dFAD_times > forcast_time_start[0]]  ## filters true dFADs locations to be inrange with dFAD forcasts 
    dFAD_times_s = dFAD_times_s[dFAD_times_s < forcast_time_start[-1]] 

    #print(dFAD_times_s/3600)
    # print(len(dFAD_times_s))
    lats = output.lat[i,:].values[mask]
    lons = output.lon[i,:].values[mask]
    lat_interp = np.interp(dFAD_times_s, forcast_time_start,lats) # interpolates Forcast times onto true dFAD times 
    lon_interp = np.interp(dFAD_times_s, forcast_time_start, lons) 
    lat_interp = np.insert(lat_interp, 0,np.nan) ## add nan at start of forcast for this is where the initial point is. 
    lon_interp = np.insert(lon_interp, 0,np.nan) ## need to add this into the true data 

    if len(dFAD_times_s) == 0: 
        emptydata += 1
        continue
    if row["geometry"] == None:
        continue
    # print(forcast_time_start)
    # print(dFAD_times)
    # print(dFAD_times_s)
    ## get index of first true point thats used in the forcast 
    idx_start = np.where(dFAD_times == dFAD_times_s[0])[0][0] ## fails at case where there is no true data 
    idx_end = np.where(dFAD_times == dFAD_times_s[-1])[0][0]
     ### could raise a probelm is idx_end is already the last value... 
    lon_true, lat_true= row["geometry"].xy
    lon_true = lon_true[idx_start-1:idx_end+1]
    lat_true = lat_true[idx_start-1:idx_end+1]
    Times = Times[idx_start-1:idx_end+1]
    leadtimes = (Times - Times[0]).astype("int64")/1e9

    Buoylist = [id]*len(lat_true) 
    dstemp= pd.DataFrame({"BuoyID": Buoylist, "Time": Times,
                            "lat_true": lat_true,"lon_true":lon_true, ""
                            "lat_forcast": lat_interp, "lon_forcast": lon_interp, 
                            "leadtime": leadtimes/3600 })
    dssave = pd.concat([dssave, dstemp])


dssave.to_csv(rf"output\Forecast{[month]}.csv")
print(emptydata)




In [ ]:
def Run_model(s,e,index): 
    print(s)
    print(e)

    return index

a = np.linspace(0,11, 12)
monthrange  = pd.date_range("2024-01-01", "2025-01-01", freq = "MS")


inputs = np.array([monthrange[:-1], monthrange[1:],np.linspace(0,11, 12) ])
inputs_r = np.rot90(inputs)


In [ ]:
import multiprocessing as mp
import numpy as np
import pandas as pd

def Run_model(s, e, index):
    print(f"Start: {s}, End: {e}, Index: {index}")
    return index


if __name__ == "__main__":  # 🔴 REQUIRED ON WINDOWS
    monthrange = pd.date_range("2024-01-01", "2025-01-01", freq="MS")

    # Build tuples of (start, end, index)
    inputs = list(zip(
        monthrange[:-1],
        monthrange[1:],
        range(len(monthrange)-1)
    ))

    with mp.Pool(processes=6) as pool:
        results = pool.starmap(Run_model, inputs)

    print("Results:", results)